# 4. Build and Evaluate the Linear Regression Model

## Purpose

This notebook builds a multiple linear regression model to predict
the number of goals scored by one team in one match.

The model uses exactly eight pre-match explanatory variables.
Match IDs, dates, stages, and team names are retained for tracking
but are not included as explanatory variables.

We evaluate the model on later matches that were not used to fit
the regression coefficients.

We also compare the model with a baseline that always predicts
the average goals scored in the training data.

In [22]:
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

# Locate the project folder.
PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent

# Load the modelling dataset created in Notebook 3.
input_path = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "worldcup2026_regression_2_2.csv"
)

assert input_path.is_file(), "Run Notebook 3 before continuing."

df = pd.read_csv(
    input_path,
    parse_dates=["match_date"]
)

# Define exactly which eight columns enter the regression model.
predictors = [
    "team_avg_goals_scored_before",
    "team_avg_goals_conceded_before",
    "team_goal_difference_before",
    "team_win_rate_before",
    "opponent_avg_goals_scored_before",
    "opponent_avg_goals_conceded_before",
    "opponent_goal_difference_before",
    "opponent_win_rate_before"
]

assert len(df) == 208
assert len(predictors) == 8
assert df[predictors + ["goals_scored"]].notna().all().all()

display(df.head())
print("Observations:", len(df))
print("Explanatory variables:", len(predictors))

,match_id,stage,match_date,team,opponent,goals_scored,team_avg_goals_scored_before,team_avg_goals_conceded_before,team_goal_difference_before,team_win_rate_before,opponent_avg_goals_scored_before,opponent_avg_goals_conceded_before,opponent_goal_difference_before,opponent_win_rate_before
0,1,Group stage,2026-06-11,Mexico,South Africa,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,1,Group stage,2026-06-11,South Africa,Mexico,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,2,Group stage,2026-06-11,Czechia,Korea Republic,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,2,Group stage,2026-06-11,Korea Republic,Czechia,2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,3,Group stage,2026-06-12,Bosnia–Herz,Canada,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


Observations: 208
Explanatory variables: 8


## 4.1 Split Earlier and Later Matches

We aim for approximately 80% of matches for training and 20% for testing.

The split is made at a date boundary. All matches on the boundary date
are assigned to the test set, so training and testing dates do not
overlap. Consequently, the exact percentages may differ from 80/20.

Both observations from the same match remain in the same partition.

Evaluation represents sequential pre-match prediction: the model
coefficients remain fixed, but a later test match may use the observed
results of earlier test matches in its historical predictors. Those
results would be available before that later match.

This is not a forecast of every remaining match made at a single
point before the entire test period.

In [23]:
# Create one chronological entry per match.
match_schedule = (
    df[["match_id", "match_date"]]
    .drop_duplicates()
    .sort_values(["match_date", "match_id"])
    .reset_index(drop=True)
)

# Find the date near the 80% match boundary.
boundary_position = int(len(match_schedule) * 0.80)
cutoff_date = match_schedule.iloc[boundary_position]["match_date"]

# Keep all matches on the cutoff date in the test partition.
train_df = df.loc[df["match_date"] < cutoff_date].copy()
test_df = df.loc[df["match_date"] >= cutoff_date].copy()

assert not train_df.empty
assert not test_df.empty

# Verify that no match appears in both partitions.
train_ids = set(train_df["match_id"])
test_ids = set(test_df["match_id"])

assert train_ids.isdisjoint(test_ids)
assert train_df["match_date"].max() < test_df["match_date"].min()

# Verify that both observations for each match stay together.
assert train_df.groupby("match_id").size().eq(2).all()
assert test_df.groupby("match_id").size().eq(2).all()

# Select only the eight explanatory variables and the target.
X_train = train_df[predictors]
y_train = train_df["goals_scored"]

X_test = test_df[predictors]
y_test = test_df["goals_scored"]

print("Test period starts:", cutoff_date.date())
print("Training matches:", train_df["match_id"].nunique())
print("Testing matches:", test_df["match_id"].nunique())
print("Training observations:", len(train_df))
print("Testing observations:", len(test_df))
print("Training percentage:", round(100 * len(train_df) / len(df), 1))
print("Testing percentage:", round(100 * len(test_df) / len(df), 1))
print("Chronological and match-separation checks passed.")

Test period starts: 2026-07-02
Training matches: 82
Testing matches: 22
Training observations: 164
Testing observations: 44
Training percentage: 78.8
Testing percentage: 21.2
Chronological and match-separation checks passed.


## 4.2 Fit the Models

The multiple linear regression estimates an intercept and eight
coefficients using only the training observations.

For comparison, the mean baseline predicts the training-set average
goals for every test observation. This helps assess whether the eight
predictors improve prediction over a simple constant estimate.

Predictions are evaluated as continuous values. We do not round them
or replace negative predictions before calculating the metrics.

In [24]:
# Fit the required multiple linear regression.
model = LinearRegression()
model.fit(X_train, y_train)

# Predict goals for the held-out later matches.
y_pred = model.predict(X_test)

# Fit a baseline using only the training-set mean.
baseline = DummyRegressor(strategy="mean")
baseline.fit(X_train, y_train)

baseline_pred = baseline.predict(X_test)

print("Linear regression fitted successfully.")
print("Training-set mean goals:", round(y_train.mean(), 3))
print("Test predictions generated:", len(y_pred))
print("Negative linear-regression predictions:", int((y_pred < 0).sum()))

Linear regression fitted successfully.
Training-set mean goals: 1.47
Test predictions generated: 44
Negative linear-regression predictions: 9


## 4.3 Evaluate Predictive Performance

We report three test-set metrics:

- R²: compares prediction error with variation around the test-set mean.
  Higher values are better. A negative value means the model performs
  worse than predicting the test-set mean.
- MAE: average absolute prediction error, measured in goals.
  Lower values are better.
- RMSE: prediction error measured in goals, with greater penalties
  for larger errors. Lower values are better.

We compare the regression with the training-mean baseline on the
same test observations. We also inspect training performance for
evidence of a large training-to-test performance gap.

In [25]:
# Calculate consistent metrics for each model and partition.
def evaluate_predictions(model_name, partition, actual, predicted):
    return {
        "model": model_name,
        "partition": partition,
        "R2": r2_score(actual, predicted),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted))
    }


train_pred = model.predict(X_train)

metrics = pd.DataFrame([
    evaluate_predictions(
        "Linear regression", "Training", y_train, train_pred
    ),
    evaluate_predictions(
        "Linear regression", "Test", y_test, y_pred
    ),
    evaluate_predictions(
        "Training-mean baseline", "Test", y_test, baseline_pred
    )
])

display(metrics.round(3))

# Save metrics without rounding the stored values.
metrics.to_csv(
    PROJECT_ROOT / "data" / "processed" / "regression_2_2_metrics.csv",
    index=False
)

,model,partition,R2,MAE,RMSE
0,Linear regression,Training,0.123,1.034,1.318
1,Linear regression,Test,-0.705,1.246,1.727
2,Training-mean baseline,Test,-0.002,1.065,1.324


## 4.4 Examine the Regression Coefficients

Each coefficient describes the change in predicted goals associated
with a one-unit increase in that predictor, holding the other
predictors constant.

For example, a one-unit increase in average goals scored means one
additional historical goal per match. A one-unit increase in win rate
means an increase from 0 to 1, or 100 percentage points.

These coefficients describe conditional associations, not causal
effects. Their magnitudes cannot be directly compared as measures
of importance because the variables use different scales.

Related form variables may be strongly correlated, making individual
coefficients unstable. We check matrix rank and will examine predictor
correlations in the visualisation notebook.

In [26]:
# Store the fitted intercept and eight coefficients.
coefficients = pd.DataFrame({
    "term": ["intercept"] + predictors,
    "coefficient": [model.intercept_] + list(model.coef_)
})

display(coefficients.round(4))

# Check whether the training design matrix has full column rank.
design_matrix = np.column_stack([
    np.ones(len(X_train)),
    X_train.to_numpy(dtype=float)
])

matrix_rank = np.linalg.matrix_rank(design_matrix)

print("Design-matrix rank:", matrix_rank)
print("Design-matrix columns:", design_matrix.shape[1])

if matrix_rank < design_matrix.shape[1]:
    print(
        "WARNING: Exact linear dependence is present. "
        "Individual coefficients are not uniquely identifiable."
    )
else:
    print(
        "No exact linear dependence detected. "
        "Strong correlations may still affect coefficient stability."
    )

coefficients.to_csv(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "regression_2_2_coefficients.csv",
    index=False
)

,term,coefficient
0,intercept,1.5516
1,team_avg_goals_scored_before,0.3014
2,team_avg_goals_conceded_before,-0.2985
3,team_goal_difference_before,0.0293
4,team_win_rate_before,-1.1275
5,opponent_avg_goals_scored_before,0.3185
6,opponent_avg_goals_conceded_before,-0.0381
7,opponent_goal_difference_before,-0.1888
8,opponent_win_rate_before,-0.3687


Design-matrix rank: 9
Design-matrix columns: 9
No exact linear dependence detected. Strong correlations may still affect coefficient stability.


## 4.5 Save Test Predictions and Partition Information

We save the actual goals, predicted goals, baseline predictions,
and residuals for each test observation.

Residual is defined as actual goals minus predicted goals.

These saved predictions allow visualisations to use exactly the same
held-out observations as the model evaluation, without fitting a
different model or creating a different split.

In [27]:
# Retain identifiers so each prediction can be traced to its match.
predictions = test_df[[
    "match_id",
    "stage",
    "match_date",
    "team",
    "opponent",
    "goals_scored"
]].copy()

predictions = predictions.rename(
    columns={"goals_scored": "actual_goals"}
)

# Predictions follow the same row order as X_test.
predictions["predicted_goals"] = y_pred
predictions["baseline_predicted_goals"] = baseline_pred
predictions["residual"] = (
    predictions["actual_goals"] - predictions["predicted_goals"]
)

predictions.to_csv(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "regression_2_2_test_predictions.csv",
    index=False
)

# Save the partition assigned to each observation.
partitions = df[[
    "match_id", "match_date", "team"
]].copy()

partitions["partition"] = np.where(
    partitions["match_date"] < cutoff_date,
    "Training",
    "Test"
)

partitions.to_csv(
    PROJECT_ROOT
    / "data"
    / "processed"
    / "regression_2_2_partitions.csv",
    index=False
)

display(predictions.head(10))

print("Saved evaluation metrics.")
print("Saved regression coefficients.")
print("Saved test predictions and residuals.")
print("Saved training/test partition information.")

,match_id,stage,match_date,team,opponent,actual_goals,predicted_goals,baseline_predicted_goals,residual
164,83,Round of 32,2026-07-02,Austria,Spain,0,0.522624,1.469512,-0.522624
165,83,Round of 32,2026-07-02,Spain,Austria,3,1.886763,1.469512,1.113237
166,84,Round of 32,2026-07-02,Croatia,Portugal,1,0.362193,1.469512,0.637807
167,84,Round of 32,2026-07-02,Portugal,Croatia,2,2.047195,1.469512,-0.047195
168,85,Round of 32,2026-07-02,Algeria,Switzerland,0,0.627044,1.469512,-0.627044
169,85,Round of 32,2026-07-02,Switzerland,Algeria,2,2.018562,1.469512,-0.018562
170,86,Round of 32,2026-07-03,Australia,Egypt,1,1.169961,1.469512,-0.169961
171,86,Round of 32,2026-07-03,Egypt,Australia,1,1.502254,1.469512,-0.502254
172,87,Round of 32,2026-07-03,Argentina,Cabo Verde,3,1.520466,1.469512,1.479534
173,87,Round of 32,2026-07-03,Cabo Verde,Argentina,2,0.699968,1.469512,1.300032


Saved evaluation metrics.
Saved regression coefficients.
Saved test predictions and residuals.
Saved training/test partition information.


## 4.6 Evaluation Summary

The chronological split assigned 82 matches (164 observations) to
training and 22 matches (44 observations) to testing.

On the test set, linear regression produced R² = -0.705,
MAE = 1.246 goals, and RMSE = 1.727 goals.

The training-mean baseline produced MAE = 1.065 goals and
RMSE = 1.324 goals. Therefore, the regression did not improve
prediction over this simple baseline on the held-out matches.

The design matrix had full column rank. However, predictor
correlations and residual patterns require further examination
before interpreting individual coefficients confidently.